# ЛР-03: Модернизация ремонтной базы

## Student notebook: military 02

Этот notebook предназначен для самостоятельного анализа.

Готовых численных ответов и заполненного sensitivity-разбора здесь нет.

## 1. Зачем нужен этот кейс

Нужно выбрать масштабы модернизации при ограниченном бюджете, персонале и мощностях.

После выполнения работы студент должен уметь:

1. читать чувствительность через нехватку ресурсов, а не только через числа;
2. аккуратно формулировать двойственную задачу;
3. проводить минимум два сценария по `b` и один по `c`;
4. делать вывод о самой дефицитной мощности.

## 2. Исходные данные

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 90 |
| Трудозатраты | 54 |
| Производственная ёмкость | 44 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Диагностические посты | 80 | 28 | 14 | 12 |
| Склад критических узлов | 74 | 24 | 11 | 10 |
| Мобильные ремонтные бригады | 88 | 36 | 21 | 16 |
| Испытательный участок | 92 | 44 | 24 | 18 |

## 3. Что нужно сделать

1. Запишите прямую модель через переменные масштабов программ.
2. Подготовьте `c`, `A_ub`, `b_ub`, `bounds` для `linprog`.
3. Определите активные ограничения и запас ресурса.
4. Кратко запишите двойственную модель.
5. Проведите минимум два сценария по `b` и один сценарий по `c`.
6. Сравните прогноз по теневой цене с фактическим пересчётом.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Шаг 1. Задаем исходные данные прямой задачи.

effects = np.array(
    [
        80,
        74,
        88,
        92,
    ],
    dtype=float,
)

A_ub = np.array(
    [
        [28, 24, 36, 44],
        [14, 11, 21, 24],
        [12, 10, 16, 18],
    ],
    dtype=float,
)

b_ub = np.array(
    [
        90,
        54,
        44,
    ],
    dtype=float,
)

bounds = [(0, 1)] * len(effects)

# Шаг 2. Формируем нейтральные подписи для табличной проверки данных.
program_names = [f"Программа {idx + 1}" for idx in range(len(effects))]
resource_names = [f"Ресурс {idx + 1}" for idx in range(len(b_ub))]

effects_df = pd.DataFrame(
    {
        "программа": program_names,
        "эффект на единицу": effects,
    }
)
A_ub_df = pd.DataFrame(
    A_ub,
    index=resource_names,
    columns=program_names,
)
b_ub_df = pd.DataFrame(
    {
        "ресурс": resource_names,
        "лимит": b_ub,
    }
)

print("Число программ =", len(effects))
print("Число ресурсных ограничений =", len(b_ub))
print("Вектор эффектов:")
display(effects_df)

print("Матрица ресурсных коэффициентов A_ub:")
display(A_ub_df)

print("Вектор правых частей b_ub:")
display(b_ub_df)

## 4. Шаблон для самостоятельной сборки решения

Сначала решите прямую задачу, потом переходите к binding/slack, dual и sensitivity.

In [ ]:
# TODO: реализуйте helper-функции после ручной записи прямой и двойственной моделей.
def solve_primal(effects, A_ub, b_ub, bounds):
    """Решает прямую задачу максимизации через `linprog`.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ.
        A_ub (np.ndarray): Матрица расхода ресурсов по программам.
        b_ub (np.ndarray): Вектор доступных лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных `0 <= x <= 1`.

    Возвращает:
        tuple: Пара `result, shadow_prices` после решения прямой задачи.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: задайте `c = -effects` и вызовите `linprog` для прямой задачи.
    return None, None


def solve_dual(effects, A_ub, b_ub):
    """Собирает и решает двойственную задачу для модели с верхними границами.

    Аргументы:
        effects (np.ndarray): Вектор эффектов прямой задачи.
        A_ub (np.ndarray): Матрица ресурсных ограничений прямой задачи.
        b_ub (np.ndarray): Вектор правых частей ресурсных ограничений.

    Возвращает:
        scipy.optimize.OptimizeResult | None: Результат решения двойственной задачи.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: соберите `c_dual`, `A_dual`, `b_dual` и границы двойственных переменных.
    return None


def rerun_with_resource_change(effects, A_ub, b_ub, bounds, resource_index, delta):
    """Пересчитывает модель после изменения одного ресурсного лимита.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ.
        A_ub (np.ndarray): Матрица расхода ресурсов.
        b_ub (np.ndarray): Исходный вектор лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных прямой задачи.
        resource_index (int): Индекс ресурса, который меняется в сценарии.
        delta (float): Приращение правой части выбранного ограничения.

    Возвращает:
        tuple | None: Новый вектор лимитов и результат повторного решения модели.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    # TODO: скопируйте `b_ub`, измените один ресурс и снова решите прямую задачу.
    return None


# TODO: подготовьте задачу максимизации через минимизацию отрицательной цели.
c = None
result = None
dual_result = None
shadow_prices = None
slack = None
binding = None

# TODO: после своей попытки решите модель и заполните анализ.
# c = -effects
# result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
# shadow_prices = -result.ineqlin.marginals
# slack = result.slack
# binding = np.isclose(slack, 0.0)
# dual_result = solve_dual(effects, A_ub, b_ub)
# np.allclose(-result.fun, dual_result.fun)

## 5. Что должно быть в отчёте

1. Прямая постановка задачи.
2. Оптимальный план по программам.
3. Таблица активных ограничений и запасов.
4. Таблица теневых цен ресурсов.
5. Проверка сильной двойственности.
6. Минимум два сценария по `b` и один по `c`.

## 6. Контрольный чек-лист

- [ ] Я показал, какие ограничения стали binding.
- [ ] Я осмысленно интерпретировал shadow prices.
- [ ] Я сравнил прогноз и фактический пересчёт.
- [ ] Я сформулировал управленческий вывод по самому дефицитному ресурсу.